# DTFA + YOLOv12n — fog L/M/H và clear

Repo: https://github.com/NguyenDoKhaiHoan/Yolv12x

Trong Kaggle Notebook: bật **GPU** và **Internet**, rồi **Add Input** dataset chứa
`fog/fog/data.yaml`, `train`, `val`, `test`, `clear images/vesion2`.
Giữ nguyên thư mục và tên file khi upload. Không chia lại L/M/H ngẫu nhiên:
ba mức của cùng cảnh phải nằm cùng split.

Notebook tạo Python 3.11 riêng trong `/kaggle/working`, không thay PyTorch của kernel.
Ảnh nằm trong `/kaggle/input`; annotation, model và metrics ghi vào `/kaggle/working`.
**AWD cần checkpoint IRT đã pretrain của DTFA.** Baseline và SPT không cần IRT.
Không dùng teacher ngẫu nhiên để thay checkpoint thiếu.

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO = Path('/kaggle/working/Yolv12x')
if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/NguyenDoKhaiHoan/Yolv12x.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
PY = '/kaggle/working/dtfa-env/bin/python'
if not Path(PY).exists():
    subprocess.run([sys.executable, '-m', 'uv', 'venv', '--python', '3.11', '--seed',
                    '/kaggle/working/dtfa-env'], check=True)
subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--python', PY,
               '--index-url', 'https://download.pytorch.org/whl/cu121',
               'torch==2.2.2', 'torchvision==0.17.2'], check=True)
subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--python', PY,
               '-r', str(REPO / 'requirements-yolov12.txt')], check=True)
ENV = dict(os.environ, YOLO_CONFIG_DIR='/kaggle/working/yolo-config',
           MPLCONFIGDIR='/kaggle/working/matplotlib')
def run(*args):
    subprocess.run([PY, *map(str, args)], cwd=REPO, env=ENV, check=True)
run('-c', "import torch; print(torch.__version__); assert torch.cuda.is_available(), 'Enable GPU'; print(torch.cuda.get_device_name(0))")

## Cấu hình
Mặc định **YOLOv12n**, trộn cả L/M/H khi train và giữ riêng mỗi mức khi test.
Đặt `EPOCHS=2` cho lần chạy thử rồi chọn output directory mới cho lần train thật.
Batch 4 là điểm bắt đầu; giảm còn 2 nếu hết VRAM. Chỉ dùng một GPU (`cuda:0`).
Nếu auto-detect có nhiều dataset, điền `DATA_ROOT` vào thư mục chứa `data.yaml`.
`IRT` là đường dẫn `.pth` đã thêm vào Kaggle Input; để `None` nếu chỉ chạy baseline/SPT.

In [ ]:
DATA_ROOT = None  # ví dụ Path('/kaggle/input/your-dataset/fog/fog')
IRT = None        # ví dụ Path('/kaggle/input/dtfa-weights/IRT.pth')
EPOCHS = 100
BATCH = 4
IMAGE_SIZE = 640
LEVELS = ['L', 'M', 'H']
RUNS = Path('/kaggle/working/runs/yolov12n')
PREPARED = Path('/kaggle/working/prepared')

if DATA_ROOT is None:
    candidates = [p.parent for p in Path('/kaggle/input').rglob('data.yaml')
                  if (p.parent / 'clear images' / 'vesion2').is_dir()]
    if len(candidates) != 1:
        raise ValueError(f'Set DATA_ROOT explicitly; found {candidates}')
    DATA_ROOT = candidates[0]
DATA_ROOT = Path(DATA_ROOT)
print('Dataset:', DATA_ROOT)
run('-m', 'dtfa12.prepare_data', '--root', DATA_ROOT, '--output', PREPARED,
    '--levels', *LEVELS)

## Trọng số khởi tạo YOLOv12n
Tải từ release **turbo** của tác giả, tương ứng kiến trúc được pin trong requirements.
Checkpoint YOLOX hoặc YOLOv12s không thay thế được YOLOv12n.

In [ ]:
import urllib.request
WEIGHTS = Path('/kaggle/working/yolov12n.pt')
if not WEIGHTS.is_file():
    temporary = WEIGHTS.with_suffix('.download')
    urllib.request.urlretrieve('https://github.com/sunsmarterjie/yolov12/releases/download/turbo/yolov12n.pt', temporary)
    temporary.replace(WEIGHTS)
COMMON = ['--scale', 'n', '--classes', PREPARED / 'classes.txt', '--weights', WEIGHTS,
          '--epochs', EPOCHS, '--batch', BATCH, '--imgsz', IMAGE_SIZE,
          '--device', 'cuda:0', '--workers', 2]

## 1. Baseline YOLOv12n trên fog
Baseline giúp đo phần cải thiện do DTFA. Có thể bỏ qua cell này nếu chỉ cần train AWD.

In [ ]:
run('-m', 'dtfa12.train', '--stage', 'baseline', *COMMON,
    '--train', PREPARED / 'train_fog.jsonl', '--val', PREPARED / 'val_fog.jsonl',
    '--output', RUNS / 'baseline')

## 2. SPT YOLOv12n trên clear
Mỗi ảnh clear chỉ dùng một lần trong tập SPT (không lặp ba lần theo L/M/H).

In [ ]:
run('-m', 'dtfa12.train', '--stage', 'spt', *COMMON,
    '--train', PREPARED / 'train_clear.jsonl', '--val', PREPARED / 'val_clear.jsonl',
    '--output', RUNS / 'spt')

## 3. DTFA–YOLOv12n trên các cặp fog/clear
Tải IRT từ liên kết pretrained trong README repo DTFA gốc và thêm vào Kaggle Input.
Mã IRT pretraining của upstream còn thiếu module; notebook dùng checkpoint IRT có sẵn.
Cell này dừng rõ ràng nếu chưa có IRT. Chọn file tương thích encoder gốc;
loader sẽ báo thiếu tensor thay vì âm thầm chạy với teacher ngẫu nhiên.

In [ ]:
if IRT is None or not Path(IRT).is_file():
    raise FileNotFoundError('Add pretrained DTFA IRT.pth to Kaggle Input and set IRT in the configuration cell.')
run('-m', 'dtfa12.train', '--stage', 'awd', *COMMON,
    '--train', PREPARED / 'train_fog.jsonl',
    '--clean-train', PREPARED / 'train_paired_clear.jsonl',
    '--val', PREPARED / 'val_fog.jsonl', '--spt', RUNS / 'spt' / 'best.pt',
    '--irt', IRT, '--output', RUNS / 'awd')

## 4. Đánh giá test theo L/M/H
Không dùng tập test để chọn epoch/hyperparameter. `detector.pt` được chọn bằng
validation detection loss; báo mAP50 và mAP50–95 native trên test ở đây.
Nếu mới train baseline, đặt `MODELS=['baseline']`.

In [ ]:
MODELS = ['baseline', 'awd']
for model_name in MODELS:
    for level in ['L', 'M', 'H', 'mixed']:
        run('-m', 'dtfa12.evaluate', '--weights', RUNS / model_name / 'detector.pt',
            '--data', PREPARED / f'{level}.yaml', '--split', 'test',
            '--imgsz', IMAGE_SIZE, '--batch', BATCH, '--device', '0',
            '--output', RUNS / 'evaluation' / f'{model_name}_{level}')

## 5. Lưu kết quả / tiếp tục phiên sau
`last.pt` chứa optimizer/bridges/scheduler/RNG; `best.pt` dành cho teacher hoặc resume;
`detector.pt` chỉ có student để predict/val. File `metrics.csv` ghi loss từng epoch.

Để giữ kết quả qua phiên Kaggle: lưu notebook version có output hoặc tải archive dưới đây.
Trước khi session kết thúc, cần giữ cả folder `runs`, không chỉ `detector.pt`.
Resume: khôi phục folder runs vào cùng vị trí `/kaggle/working`, cài lại môi trường,
chạy lại bước prepare và dùng **cùng lệnh train cũ** thêm
`--resume /kaggle/working/runs/yolov12n/awd/last.pt`.
Giữ nguyên epochs, batch, image size, lớp và nội dung teacher/dataset. Không tăng epochs
khi resume vì sẽ đổi lịch cosine. Đường dẫn mount teacher có thể đổi nếu nội dung giữ nguyên.

In [ ]:
import shutil
archive = shutil.make_archive('/kaggle/working/yolov12n-results', 'zip', RUNS.parent, RUNS.name)
print('Download from notebook Output:', archive)